# Keras/TensorFlow — Chapter 10: Project — Binary Classification of Sonar Returns


## 1. Bài toán và dữ liệu

- Dùng bộ **Sonar**(60 biến liên tục, phân loại M/R). Benchmark tham chiếu: **84% (trung bình), 88% (upper bound)**.

## 2. Baseline: 60→1

- Model tối giản: input 60 → hidden 60 (ReLU) → output 1 (Sigmoid). `binary_crossentropy` + `Adam`.
- Đánh giá bằng `KerasClassifier` + `StratifiedKFold(10)` + `cross_val_score()`.
- **Kết quả**: **81.68% ± 7.26%**.

## 3. Chuẩn hoá dữ liệu bằng sklearn Pipeline

Thay vì chuẩn hoá `X` một lần trước khi đưa vào k-fold, thì dùng `sklearn.pipeline.Pipeline([('standardize', StandardScaler()), ('mlp', KerasClassifier(...))])`.

- **Vì sao cách này tốt hơn:** `StandardScaler` phải được **fit lại trên riêng phần train của từng fold**, không phải fit 1 lần trên toàn bộ `X` trước khi chia fold. Nếu fit trên toàn bộ trước, thống kê (mean/std) của phần test mỗi fold sẽ rò rỉ vào bước chuẩn hoá train, nhưng sklearn `Pipeline` xử lý **tự động và đúng** cho từng fold.
- **Kết quả**: **84.56% ± 5.74%** — tăng nhẹ so với baseline, đúng benchmark 84% của sách.

## 4. Điều chỉnh kiến trúc: nhỏ hơn vs lớn hơn

- **Smaller** (60→**30**→1, giảm 1 nửa neuron hidden): ép model chỉ giữ lại đặc trưng quan trọng nhất là dữ liệu sonar có nhiều biến dư thừa. **Kết quả: 86.04% ± 4.00%** — tốt hơn baseline, và **std giảm mạnh** ổn định hơn, dù model chỉ bằng nửa kích thước.
- **Larger** (60→60→**30**→1, thêm 1 hidden layer): cho cơ hội trích đặc trưng qua nhiều tầng hơn. **Kết quả: 83.14% ± 4.52%** — **không cải thiện** so với baseline/smaller có thể do nhiễu thống kê hoặc cần train thêm epoch.
- **Bài học thực nghiệm cốt lõi**: model nhỏ hơn thắng model lớn hơn ở đây — phản bác trực giác "phức tạp hơn luôn tốt hơn". Đây chính xác là lý do nó được nhấn mạnh cần thử nghiệm và đo bằng cross-validation, không đoán bằng trực giác.

## 5. Tổng hợp 4 thí nghiệm

| Thí nghiệm | Kiến trúc | Accuracy | Std |
|---|---|---|---|
| Baseline | 60→1 | 81.68% | 7.26% |
| Standardized | 60→1 (đã chuẩn hoá) | 84.56% | 5.74% |
| Smaller | 60→30→1 (đã chuẩn hoá) | **86.04%** | **4.00%** |
| Larger | 60→60→30→1 (đã chuẩn hoá) | 83.14% | 4.52% |


## 6. Vận dụng


**10.6** — Model baseline (60→1) + đánh giá k-fold

In [1]:
# Binary Classification with Sonar Dataset: Baseline
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
# load dataset
dataframe = read_csv("sonar.csv", header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:60].astype(float)
Y = dataset[:,60]
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# baseline model
def create_baseline():
    # create model
    model = Sequential()
    model.add(Dense(60, input_shape=(60,), activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model
# evaluate model with standardized dataset
estimator = KerasClassifier(model=create_baseline, epochs=100, batch_size=5, verbose=0)
kfold = StratifiedKFold(n_splits=10, shuffle=True)
results = cross_val_score(estimator, X, encoded_Y, cv=kfold)
print("Baseline: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


2026-09-06 07:06:32.570128: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:06:32.643013: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 07:06:32.643096: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 07:06:32.645849: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 07:06:32.667873: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:06:32.668717: I tensorflow/core/platform/cpu_feature_guard.cc:1

Baseline: 83.19% (6.91%)


**10.8** — Thêm StandardScaler vào Pipeline

In [2]:
# Binary Classification with Sonar Dataset: Standardized
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("sonar.csv", header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:60].astype(float)
Y = dataset[:,60]
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# baseline model
def create_baseline():
    # create model
    model = Sequential()
    model.add(Dense(60, input_shape=(60,), activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model
# evaluate baseline model with standardized dataset
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasClassifier(model=create_baseline,
                                          epochs=100, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = StratifiedKFold(n_splits=10, shuffle=True)
results = cross_val_score(pipeline, X, encoded_Y, cv=kfold)
print("Standardized: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


Standardized: 84.64% (4.60%)


**10.10** — Mô hình nhỏ hơn (60→30→1)

In [3]:
# Binary Classification with Sonar Dataset: Standardized Smaller
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("sonar.csv", header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:60].astype(float)
Y = dataset[:,60]
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# smaller model
def create_smaller():
    # create model
    model = Sequential()
    model.add(Dense(30, input_shape=(60,), activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasClassifier(model=create_smaller,
                                          epochs=100, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = StratifiedKFold(n_splits=10, shuffle=True)
results = cross_val_score(pipeline, X, encoded_Y, cv=kfold)
print("Smaller: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


Smaller: 86.50% (5.26%)


**10.12** — Mô hình lớn hơn (60→60→30→1)

In [4]:
# Binary Classification with Sonar Dataset: Standardized Larger
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("sonar.csv", header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:60].astype(float)
Y = dataset[:,60]
# encode class values as integers
encoder = LabelEncoder()
encoder.fit(Y)
encoded_Y = encoder.transform(Y)
# larger model
def create_larger():
    # create model
    model = Sequential()
    model.add(Dense(60, input_shape=(60,), activation='relu'))
    model.add(Dense(30, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    # Compile model
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasClassifier(model=create_larger,
                                          epochs=100, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = StratifiedKFold(n_splits=10, shuffle=True)
results = cross_val_score(pipeline, X, encoded_Y, cv=kfold)
print("Larger: %.2f%% (%.2f%%)" % (results.mean()*100, results.std()*100))


Larger: 88.02% (6.43%)
